# RunPod: V4, the Sieve (the two-stage world model of `aac_recipe.md`)

**The model.** Level 1 alone, B1's encoder (ViT-tiny, CLS, projector), predictor and action embedding, with history 1, the prediction target detached, **no SIGReg, no IDM**, and a pixel decoder that reads only the latent (`sieve/decoder.py`).
- **Stage 1 (availability):** the encoder reconstructs every frame (each patch's error weighted by its own size, no labels) while the predictor learns the next latent.
- **Stage 2 (consolidation)**, forked from a stage-1 epoch with a fresh optimiser at the same rate: the predicted latent is decoded and compared with the real next frame, and the real next frame is reconstructed only where that decoded prediction was right (the gate). The latent keeps what the predictor foresees and releases what it cannot (noise).

Both stages use a constant learning rate, so a stage is **extended by resuming**: raise its epochs in the configuration cell and run its start cell again.

**How this notebook runs.** Each stage is two workers on the one GPU, started detached (a pod restart: run the start cell again, the training resumes from `lightning_resume/last.ckpt` and the metrics skip the epochs done):
1. the training (`main_hjepa.py --config-name craftax_sieve`), which saves the model at the start (epoch 0) and after every epoch;
2. the metrics worker (`python -m sieve.watch`), which reads each epoch checkpoint as it appears and rewrites the stage's report (`<run>/sieve/report.md`).

The progress cells show one bar per worker. The smoke run first times batch 128 against 256 (and the GPU use), stage 2, and one metrics check of each, so the hours are known before the real run.

**The metrics** (`sieve/check.py` has every definition; labels are only read, never trained on):
- `holds_<entity>`: the MLP probe on the latent, fitted on training worlds and scored on held-out worlds (chance in brackets).
- `holds_<entity>_linear` and `linear_vs_mlp`: what a plain distance sees.
- `foresees_<entity>` against `copy_<entity>`: the probe on the predicted latent, scored only where the entity changed.
- `recon_error_mean` and `recon_error_worst1pct`: the own decoder on held-out frames.
- `effective_rank`: the collapse guard, since SIGReg is off.
- `decoded_pred_error` and `sharp_patch_share`: the gate's own quantity.
- `recon_vs_pred_gradient`: the reconstruction's encoder gradient over the prediction's.
- `action_lift` and `action_follow*`: the action bypass.
- `moves_for_facing / mob / tile / item / noise`: what moves the latent, from frames re-rendered from stored simulator states.
- Stage 2 only, `fresh_decoder_error`: a new decoder on the frozen latent.

**When to switch and when to stop** (`sieve/report.py`; heuristics, see the recipe):
- **Stage 1 is ready** when, at two epochs in a row, no probe still rises (by more than 0.01 or half its interval), the held-out reconstruction is flat, the rank has not collapsed and the train/held-out gaps do not grow. The report proposes that epoch; you set `FORK_EPOCH`.
- **Stage 2 can stop** when the facing, the faced tile and the mobs no longer rise, the noise barely moves the latent and the rank is flat. If they still rise, extend.

Once per pod, in a terminal: `git clone https://github.com/quastAI/AAC.git /root/AAC` (the clone must contain this notebook, `sieve/` and `craftax_sieve.yaml`: push them first), then open this notebook from the clone.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
import os, shutil, sys, time
from pathlib import Path

EXPERIMENT = "V4"                           # the Sieve: one id for both stages (stage 1 in V4/, stage 2 in V4_stage2_from<E>/)

REPO = Path("/root/AAC")                    # the git clone, on the local disk (pod.py has the other paths and why they are where they are)
sys.path.insert(0, str(REPO / "notebooks"))
import pod

HF_REPO_ID = "quastAI/craftax-classic-pqn-rnn-v1"   # the dataset repo of runpod_dataset.ipynb (data/), runs go to runs/
HF_PRIVATE = True
SEEDS = "42"                                # one seed
USE_WANDB = True                            # live curves in Weights & Biases (project hjepa-flat); False = the metrics CSV only

BATCH_SIZE = 256                            # the smoke run times 128 and this one; B1 trained with 128
LEARNING_RATE = 7e-5                        # constant after a 500-step warmup (stage 2 starts at it without a warmup); B1: 5e-5 at batch 128
STAGE1_EPOCHS = 6                           # to extend stage 1: raise it and run the stage-1 start cell again (it resumes)
STAGE2_EPOCHS = 4                           # the same for stage 2
FORK_EPOCH = None                           # the stage-1 epoch stage 2 starts from: the stage-1 report proposes one, you set it
SMOKE_BATCHES = 300                         # training batches per smoke run (the rate is measured after the first 100)

In [ ]:
# --- 0. Paths, Hugging Face login, the PyTorch environment (installed again on a new pod) and the dataset ----------------
pod.configure(repo=REPO, seeds=SEEDS)
from pod import DATA, H_JEPA, LOGS, PYTHON, launch, run_dir, running, sh, show

TRAIN_OVERRIDES = pod.train_overrides(USE_WANDB)   # Hydra overrides for every training run
api = pod.login(HF_REPO_ID, HF_PRIVATE, USE_WANDB)
pod.install(api, HF_REPO_ID, USE_WANDB)
SPEC = pod.run_spec(EXPERIMENT, EXPERIMENT)        # the training config (craftax_sieve) and its overrides
FOLDER = pod.folder_of(EXPERIMENT)                 # stage 1 trains into it
SEED = SEEDS.split(",")[0]
TRAIN, VAL = DATA / "craftax_expert_train.h5", DATA / "craftax_expert_val.h5"
BANK = DATA / "sieve_counterfactuals_val.npz"      # the re-rendered frames of the moves_for_* metrics (built below, once per pod)
COMMON = [*SPEC.get("overrides", []), f"seed={SEED}", f"level1.lr={LEARNING_RATE}", f"optimizer.lr={LEARNING_RATE}", *TRAIN_OVERRIDES]
print(EXPERIMENT, FOLDER, SPEC["config"], *SPEC.get("overrides", []))

## 1. The fixed inputs of the metrics, then the smoke run

The metrics read the same frames at every epoch: 24,576 frames of training worlds (the probes and the fresh decoder are fitted there) and 16,384 of held-out worlds, cached next to the data. The counterfactual bank re-renders stored simulator states after one edit (facing, a cow, the faced tile, an item) and night frames with another noise draw. It checks that an unedited render equals the stored frame (`render_match` 1.0).

The smoke run trains `SMOKE_BATCHES` batches at batch 128 and at `BATCH_SIZE` (stage 1), then stage 2 forked from the second, and runs one full metrics pass on each (epoch 0 and 1). It prints:
- the steady steps per second, clips per second and GPU use of each batch size;
- the hours per epoch;
- the seconds per metrics check;
- the gradient ratio `recon_vs_pred_gradient`. The reconstruction weight is 0.3 (the grid's); if the ratio is far from 1 after training starts, change `level1.sieve.recon_weight` in `craftax_sieve.yaml`.

In [ ]:
# --- 1. The sample of the metrics and the counterfactual bank (once per pod; CPU) --------------------------------------------
t0 = time.time()
if not BANK.exists():
    sh([PYTHON, "-m", "sieve.counterfactuals", "--val-file", VAL, "--out", BANK], cwd=H_JEPA, quiet=True, log="sieve_bank")
show(BANK.with_suffix(".json"), 30)
sh([PYTHON, "-c", "import sys; from sieve import sample; s = sample.load(sys.argv[1], sys.argv[2]); print(len(s.train), 'training and', len(s.val), 'held-out frames')",
    TRAIN, VAL], cwd=H_JEPA)
print(f"inputs ready in {time.time() - t0:.0f} s")

In [ ]:
# --- 1. Smoke run: 128 against BATCH_SIZE (stage 1), stage 2, one metrics pass of each (full logs in LOGS/V4_smoke_*.log) ------------
import h5py, json
SMOKE = f"{EXPERIMENT}_smoke"
SMOKE_ARGS = [*COMMON, "trainer.max_epochs=1", f"+trainer.limit_train_batches={SMOKE_BATCHES}", "+trainer.limit_val_batches=2",
              "resume_every_n_steps=0", "wandb.enabled=false"]                     # no W&B run for a smoke run
with h5py.File(TRAIN, "r") as f:
    clips = int((f["ep_len"][:].astype(int) - 1).clip(min=0).sum())               # the starts of the 2-frame clips of one epoch
timing = {}
for batch in sorted({128, BATCH_SIZE}):
    name = f"{SMOKE}_b{batch}"
    shutil.rmtree(run_dir(name), ignore_errors=True)
    with pod.GpuSampler() as gpu:
        sh([PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *SMOKE_ARGS, f"output_model_name={name}", f"loader.batch_size={batch}"],
           cwd=H_JEPA, quiet=True, log=name)
    rate = pod.steady_rate(LOGS / f"{name}.log")
    timing[f"stage 1, batch {batch}"] = dict(batch=batch, rate=rate, util=gpu.utilization, memory=gpu.memory)

stage1 = run_dir(f"{SMOKE}_b{BATCH_SIZE}")
name = f"{SMOKE}_stage2"
shutil.rmtree(run_dir(name), ignore_errors=True)
with pod.GpuSampler() as gpu:
    sh([PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *SMOKE_ARGS, f"output_model_name={name}", f"loader.batch_size={BATCH_SIZE}",
        "level1.sieve.stage=2", f"init_from={stage1 / f'{SMOKE}_b{BATCH_SIZE}_epoch_1_object.ckpt'}", "optimizer.warmup_steps=0"],
       cwd=H_JEPA, quiet=True, log=name)
timing[f"stage 2, batch {BATCH_SIZE}"] = dict(batch=BATCH_SIZE, rate=pod.steady_rate(LOGS / f"{name}.log"), util=gpu.utilization, memory=gpu.memory)

checks = {}
for run in (stage1, run_dir(name)):
    sh([PYTHON, "-m", "sieve.watch", "--run", run, "--train-file", TRAIN, "--val-file", VAL, "--bank", BANK, "--once"],
       cwd=H_JEPA, quiet=True, log=f"{run.parent.name}_metrics")
    checks[run.parent.name] = [json.loads((run / "sieve" / f"epoch_{e}.json").read_text()) for e in (0, 1)]

print(f"{clips:,} clips per epoch\n")
print(f"{'':22s} {'steps/s':>8s} {'clips/s':>8s} {'GPU %':>6s} {'GPU MiB':>8s} {'h/epoch':>8s}")
for label, t in timing.items():
    hours = clips / t["batch"] / t["rate"] / 3600 if t["rate"] else float("nan")
    t["hours"] = hours
    print(f"{label:22s} {t['rate'] or float('nan'):8.2f} {(t['rate'] or 0) * t['batch']:8.0f} {t['util'] or float('nan'):6.0f} {t['memory'] or float('nan'):8.0f} {hours:8.2f}")
s1, s2 = timing[f"stage 1, batch {BATCH_SIZE}"], timing[f"stage 2, batch {BATCH_SIZE}"]
m1, m2 = checks[f"{SMOKE}_b{BATCH_SIZE}"], checks[f"{SMOKE}_stage2"]
print(f"\nmetrics check: {m1[1]['seconds']:.0f} s per epoch in stage 1, {m2[1]['seconds']:.0f} s in stage 2 (in parallel with the training)")
print(f"estimate: stage 1 {STAGE1_EPOCHS} epochs {STAGE1_EPOCHS * s1['hours']:.1f} h, stage 2 {STAGE2_EPOCHS} epochs {STAGE2_EPOCHS * s2['hours']:.1f} h "
      "(the training only: the per-epoch health evaluation and the second worker on the same GPU add a little)")
print(f"recon_vs_pred_gradient (stage 1, recon_weight 0.3): start {m1[0].get('recon_vs_pred_gradient', float('nan')):.2f}, "
      f"after {SMOKE_BATCHES} steps {m1[1].get('recon_vs_pred_gradient', float('nan')):.2f}; stage 2: {m2[1].get('recon_vs_pred_gradient', float('nan')):.2f}")

## 2. Stage 1 (availability)

Starts the training and the metrics worker, detached. To **extend**, raise `STAGE1_EPOCHS` and run this cell again: the training resumes from its last state (constant rate) and the worker evaluates only the new epochs. The stage-1 epoch checkpoints stay on the volume (not uploaded): stage 2 forks from them.

In [ ]:
# --- 2. Start (or resume, or extend) stage 1: the training and the metrics worker ---------------------------------------------
TRAIN1, METRICS1 = f"{EXPERIMENT}_stage1", f"{EXPERIMENT}_stage1_metrics"
STAGE1_RUN = run_dir(FOLDER)
launch(TRAIN1, [PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *COMMON, f"output_model_name={FOLDER}",
                f"loader.batch_size={BATCH_SIZE}", f"trainer.max_epochs={STAGE1_EPOCHS}", f"wandb.config.name={EXPERIMENT}_stage1"], cwd=H_JEPA)
launch(METRICS1, [PYTHON, "-m", "sieve.watch", "--run", STAGE1_RUN, "--train-file", TRAIN, "--val-file", VAL, "--bank", BANK,
                  "--until-epoch", STAGE1_EPOCHS, "--training-pid-file", LOGS / f"{TRAIN1}.pid"], cwd=H_JEPA)

In [ ]:
# --- 2. Progress of both workers (interrupt any time: the jobs keep running; run it again to follow them) -----------------------
pod.follow(pod.sieve_bars(TRAIN1, METRICS1, STAGE1_RUN, STAGE1_EPOCHS))
print("\n".join(pod.progress(TRAIN1, "[health", 2) + pod.progress(METRICS1, "[sieve]", 3)))

In [ ]:
# --- 2. Loss curves of stage 1 (also while it trains) and the stage-1 report: the readiness of every epoch, the proposed fork --------
from IPython.display import Image, display
sh([PYTHON, REPO / "analysis" / "plot_training.py", "--run", STAGE1_RUN], check=False)
if (STAGE1_RUN / "health" / "loss_curves.png").exists():
    display(Image(filename=str(STAGE1_RUN / "health" / "loss_curves.png")))
show(STAGE1_RUN / "sieve" / "report.md", 200)
if (STAGE1_RUN / "sieve" / "report.png").exists():
    display(Image(filename=str(STAGE1_RUN / "sieve" / "report.png")))
proposal = json.loads((STAGE1_RUN / "sieve" / "report.json").read_text()).get("ready_epoch") if (STAGE1_RUN / "sieve" / "report.json").exists() else None
print("proposed FORK_EPOCH:", proposal if proposal is not None else "none yet (extend stage 1 if the probes still rise)")

## 3. Stage 2 (consolidation)

Set `FORK_EPOCH` in the configuration cell, usually the stage-1 report's proposal, and run that cell again. Stage 2 trains into `V4_stage2_from<FORK_EPOCH>/`. It starts with the stage-1 weights, a fresh optimiser and the same rate, without a warmup. Its epoch 0 is the parent. Several forks, from different stage-1 epochs, can exist side by side (the recipe's later check of the switch time). To **extend**, raise `STAGE2_EPOCHS` and run the start cell again.

In [ ]:
# --- 3. Start (or resume, or extend) stage 2 from FORK_EPOCH: the training and the metrics worker ------------------------------------
assert FORK_EPOCH is not None, "set FORK_EPOCH in the configuration cell (the stage-1 report proposes one) and run that cell again"
STAGE2_FOLDER = f"{FOLDER}_stage2_from{FORK_EPOCH}"
STAGE2_RUN = run_dir(STAGE2_FOLDER)
parent = run_dir(FOLDER) / f"{FOLDER}_epoch_{FORK_EPOCH}_object.ckpt"
assert parent.exists(), f"{parent} is missing: stage 1 has not reached epoch {FORK_EPOCH} on this volume"
TRAIN2, METRICS2 = f"{EXPERIMENT}_stage2_from{FORK_EPOCH}", f"{EXPERIMENT}_stage2_from{FORK_EPOCH}_metrics"
launch(TRAIN2, [PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *COMMON, f"output_model_name={STAGE2_FOLDER}",
                f"loader.batch_size={BATCH_SIZE}", f"trainer.max_epochs={STAGE2_EPOCHS}", "level1.sieve.stage=2", f"init_from={parent}",
                "optimizer.warmup_steps=0", f"wandb.config.name={TRAIN2}"], cwd=H_JEPA)
launch(METRICS2, [PYTHON, "-m", "sieve.watch", "--run", STAGE2_RUN, "--train-file", TRAIN, "--val-file", VAL, "--bank", BANK,
                  "--until-epoch", STAGE2_EPOCHS, "--training-pid-file", LOGS / f"{TRAIN2}.pid"], cwd=H_JEPA)

In [ ]:
# --- 3. Progress of both workers ----------------------------------------------------------------------------------------------------
pod.follow(pod.sieve_bars(TRAIN2, METRICS2, STAGE2_RUN, STAGE2_EPOCHS))
print("\n".join(pod.progress(TRAIN2, "[health", 2) + pod.progress(METRICS2, "[sieve]", 3)))

In [ ]:
# --- 3. Loss curves of stage 2 and the stage-2 report: whether it can stop ------------------------------------------------------------
from IPython.display import Image, display
sh([PYTHON, REPO / "analysis" / "plot_training.py", "--run", STAGE2_RUN], check=False)
if (STAGE2_RUN / "health" / "loss_curves.png").exists():
    display(Image(filename=str(STAGE2_RUN / "health" / "loss_curves.png")))
show(STAGE2_RUN / "sieve" / "report.md", 200)
if (STAGE2_RUN / "sieve" / "report.png").exists():
    display(Image(filename=str(STAGE2_RUN / "sieve" / "report.png")))

## 3b. Stage 2 with the diff gate

The same fork from `FORK_EPOCH` with `level1.sieve.gate=diff`: the gate compares the decoded prediction with the decoded latent of the next frame instead of the pixels (`sieve/losses.py`). The recipe's gate (`abs`) also shuts patches the decoder cannot draw within tau where the prediction is right: on stage-1 epoch 8 it shut the ores, the table and the furnace for that reason (`sieve/gate_analysis.py`, `own_ok` about equal to `sharp`). The diff gate opens them (about 0.1 to 0.6 at tau) and still keeps entering and changed tiles at about half the unchanged ones. It does not shut night noise the latent does not hold: watch `moves_for_noise`.

It trains into `V4_stage2diff_from<FORK_EPOCH>/`, next to the abs fork. The start cell points `STAGE2_RUN`, `TRAIN2` and `METRICS2` at this run, so the progress and report cells of section 3 then follow it. On a new pod: the configuration and paths cells and the bank cell of section 1 first, and stage 1's `V4_epoch_<FORK_EPOCH>_object.ckpt` must be on the volume (the Hub does not have the epoch checkpoints).

In [ ]:
# --- 3b. Start (or resume, or extend) stage 2 with the diff gate from FORK_EPOCH; then the progress and report cells of section 3 -----
GATE = "diff"
assert FORK_EPOCH is not None, "set FORK_EPOCH in the configuration cell (the stage-1 report proposes one) and run that cell again"
STAGE2_FOLDER = f"{FOLDER}_stage2{GATE}_from{FORK_EPOCH}"
STAGE2_RUN = run_dir(STAGE2_FOLDER)
parent = run_dir(FOLDER) / f"{FOLDER}_epoch_{FORK_EPOCH}_object.ckpt"
assert parent.exists(), f"{parent} is missing: stage 1's epoch {FORK_EPOCH} is not on this volume (copy it there, the Hub has no epoch checkpoints)"
TRAIN2, METRICS2 = f"{EXPERIMENT}_stage2{GATE}_from{FORK_EPOCH}", f"{EXPERIMENT}_stage2{GATE}_from{FORK_EPOCH}_metrics"
launch(TRAIN2, [PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *COMMON, f"output_model_name={STAGE2_FOLDER}",
                f"loader.batch_size={BATCH_SIZE}", f"trainer.max_epochs={STAGE2_EPOCHS}", "level1.sieve.stage=2", f"level1.sieve.gate={GATE}",
                f"init_from={parent}", "optimizer.warmup_steps=0", f"wandb.config.name={TRAIN2}"], cwd=H_JEPA)
launch(METRICS2, [PYTHON, "-m", "sieve.watch", "--run", STAGE2_RUN, "--train-file", TRAIN, "--val-file", VAL, "--bank", BANK,
                  "--until-epoch", STAGE2_EPOCHS, "--training-pid-file", LOGS / f"{TRAIN2}.pid"], cwd=H_JEPA)

## 4. Upload

The runs (stage 1 and the stage-2 forks that exist) go to `runs/` of the Hub repo: the final models, `sieve/` with every epoch's metrics, the health reports and the configs. The epoch checkpoints and the resumable state stay on the volume. The reports go to `reports/` as `V4_stage1.md`, `V4_stage2_from<E>.md` and `V4_stage2diff_from<E>.md`.

In [ ]:
# --- 4. The reports and both runs to the Hub ------------------------------------------------------------------------------------------
folders, reports = [FOLDER], {"stage1": run_dir(FOLDER)}
for tag in ("stage2", "stage2diff"):                       # the abs fork and the diff-gate fork, those that exist
    if FORK_EPOCH is not None and run_dir(f"{FOLDER}_{tag}_from{FORK_EPOCH}").exists():
        folders.append(f"{FOLDER}_{tag}_from{FORK_EPOCH}")
        reports[f"{tag}_from{FORK_EPOCH}"] = run_dir(folders[-1])
for tag, run in reports.items():
    for suffix in (".md", ".png"):
        if (run / "sieve" / f"report{suffix}").exists():   # copyfile: the volume refuses the file times copy2 sets
            shutil.copyfile(run / "sieve" / f"report{suffix}", DATA / "reports" / f"{EXPERIMENT}_{tag}{suffix}")
pod.upload(api, HF_REPO_ID, folders=folders, reports=[f"{EXPERIMENT}_*.md", f"{EXPERIMENT}_*.png"])